In [2]:
!pip install pytorch-tabnet -q
import os
import gc
import random
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch

from pytorch_tabnet.tab_model import TabNetRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [ ]:
#Seed 고정
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

print("SEED:", SEED)

SEED: 42


In [ ]:

FILE_PATH = "/Users/seomichelle/26-1 ESAA:Python/datasets/Numerai/pca500_lag1_lag2.parquet"

OUTPUT_PATH = "/Users/seomichelle/26-1 ESAA:Python/datasets/Numerai/tabnet_val_prediction.csv"

In [ ]:
#  Parquet Metadata 확인


parquet_file = pq.ParquetFile(FILE_PATH)
all_cols = parquet_file.schema_arrow.names

print("전체 row 수:", parquet_file.metadata.num_rows)
print("전체 컬럼 수:", len(all_cols))
print("앞 컬럼 10개:", all_cols[:10])
print("뒤 컬럼 10개:", all_cols[-10:])

전체 row 수: 587583
전체 컬럼 수: 1503
앞 컬럼 10개: ['pca_0', 'pca_1', 'pca_2', 'pca_3', 'pca_4', 'pca_5', 'pca_6', 'pca_7', 'pca_8', 'pca_9']
뒤 컬럼 10개: ['pca_493_lag2', 'pca_494_lag2', 'pca_495_lag2', 'pca_496_lag2', 'pca_497_lag2', 'pca_498_lag2', 'pca_499_lag2', 'era', 'target', 'prediction']


In [ ]:
# Feature / Target 설정


base_cols = [
    c for c in all_cols
    if c.startswith("pca_") and not c.endswith("_lag1") and not c.endswith("_lag2")
]

lag1_cols = [
    c for c in all_cols
    if c.startswith("pca_") and c.endswith("_lag1")
]

lag2_cols = [
    c for c in all_cols
    if c.startswith("pca_") and c.endswith("_lag2")
]

FEATURES = base_cols + lag1_cols + lag2_cols

print("base PCA 개수:", len(base_cols))
print("lag1 개수:", len(lag1_cols))
print("lag2 개수:", len(lag2_cols))
print("총 feature 개수:", len(FEATURES))
if len(FEATURES) != 1500:
    print("주의: feature 개수가 1500개가 아닙니다. 컬럼명을 다시 확인하세요.")

# target_cyrusd_20이 원본명이고,
# 전처리된 parquet에서는 target으로 저장되어 있을 가능성이 높음
if "target_cyrusd_20" in all_cols:
    TARGET_COL = "target_cyrusd_20"
elif "target" in all_cols:
    TARGET_COL = "target"
else:
    raise ValueError("target_cyrusd_20 또는 target 컬럼이 없습니다.")

ERA_COL = "era"

if ERA_COL not in all_cols:
    raise ValueError("era 컬럼이 없습니다.")

print("실제 사용 target column:", TARGET_COL)
print("원래 통일 target 기준: target_cyrusd_20")


base PCA 개수: 500
lag1 개수: 500
lag2 개수: 500
총 feature 개수: 1500
실제 사용 target column: target
원래 통일 target 기준: target_cyrusd_20


In [ ]:
use_cols = FEATURES + [ERA_COL, TARGET_COL]

df = pd.read_parquet(FILE_PATH, columns=use_cols)

df[FEATURES] = (
    df[FEATURES]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
    .astype("float32")
)

df[TARGET_COL] = df[TARGET_COL].astype("float32")

df = df[df[TARGET_COL].notna()].copy()

print("df shape:", df.shape)
display(df.head())

gc.collect()


df shape: (587583, 1502)


,pca_0,pca_1,pca_2,pca_3,pca_4,pca_5,pca_6,pca_7,pca_8,pca_9,...,pca_492_lag2,pca_493_lag2,pca_494_lag2,pca_495_lag2,pca_496_lag2,pca_497_lag2,pca_498_lag2,pca_499_lag2,era,target
0,2.157082,6.083780,-10.034975,-3.417653,-5.101716,0.658862,9.538446,-4.302553,7.393528,-2.881266,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,460,0.75
1,-6.917748,8.086533,-1.229872,1.661412,-2.063528,0.038389,-0.797849,2.407780,5.794246,9.493385,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,460,0.75
2,-9.775057,-4.105815,-7.142505,-5.733612,0.522592,-1.688702,0.250018,9.843674,7.669866,9.348386,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,460,0.50
3,-19.487883,-13.354597,-9.048046,0.205841,3.183959,8.305212,-7.106100,7.675611,8.001382,4.287366,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,460,0.75
4,7.916985,-4.583646,-7.021014,-10.736472,9.984262,-3.021521,-1.213559,-1.623956,1.568986,-6.605861,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,460,0.25


5968

In [ ]:
# Era 기준 Train / Validation Split

unique_eras = sorted(df[ERA_COL].unique())
split_point = int(len(unique_eras) * 0.8)

train_eras = unique_eras[:split_point]   # 앞 80%
valid_eras = unique_eras[split_point:]   # 뒤 20%

train_df = df[df[ERA_COL].isin(train_eras)].reset_index(drop=True)
valid_df = df[df[ERA_COL].isin(valid_eras)].reset_index(drop=True)

print("전체 era 개수:", len(unique_eras))
print("train era 개수:", len(train_eras))
print("valid era 개수:", len(valid_eras))
print("train era 범위:", min(train_eras), "~", max(train_eras))
print("valid era 범위:", min(valid_eras), "~", max(valid_eras))
print("train rows:", len(train_df))
print("valid rows:", len(valid_df))


전체 era 개수: 115
train era 개수: 92
valid era 개수: 23
train era 범위: 460 ~ 551
valid era 범위: 552 ~ 574
train rows: 461940
valid rows: 125643


In [ ]:
# X, y 생성

X_train = train_df[FEATURES].values.astype("float32")
y_train = train_df[TARGET_COL].values.astype("float32").reshape(-1, 1)

X_valid = valid_df[FEATURES].values.astype("float32")
y_valid = valid_df[TARGET_COL].values.astype("float32").reshape(-1, 1)

val_era = valid_df[ERA_COL].values
val_target = valid_df[TARGET_COL].values.astype("float32")

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_valid:", X_valid.shape)
print("y_valid:", y_valid.shape)

X_train: (461940, 1500)
y_train: (461940, 1)
X_valid: (125643, 1500)
y_valid: (125643, 1)


In [ ]:
#  TabNet 모델 정의


device_name = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device_name)

tabnet_model = TabNetRegressor(
    n_d=32,
    n_a=32,
    n_steps=5,
    gamma=1.5,
    n_independent=2,
    n_shared=2,
    lambda_sparse=1e-4,
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=2e-3),
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    scheduler_params={
        "step_size": 10,
        "gamma": 0.9
    },
    mask_type="entmax",
    seed=SEED,
    device_name=device_name,
    verbose=10
)


device: cpu


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


In [ ]:
# TabNet 

tabnet_model = TabNetRegressor(
    n_d=16,
    n_a=16,
    n_steps=3,
    gamma=1.3,
    n_independent=2,
    n_shared=2,
    lambda_sparse=1e-4,
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=2e-3),
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    scheduler_params={
        "step_size": 5,
        "gamma": 0.9
    },
    mask_type="entmax",
    seed=42,
    device_name=device_name,
    verbose=1
)

tabnet_model.fit(
    X_train=X_train,
    y_train=y_train,
    eval_set=[(X_valid, y_valid)],
    eval_name=["valid"],
    eval_metric=["rmse"],
    max_epochs=20,
    patience=3,
    batch_size=16384,
    virtual_batch_size=1024,
    num_workers=0,
    drop_last=False
)

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 3.53904 | valid_rmse: 1.38589 |  0:01:29s
epoch 1  | loss: 0.69043 | valid_rmse: 0.70117 |  0:03:00s
epoch 2  | loss: 0.29257 | valid_rmse: 0.45652 |  0:04:31s
epoch 3  | loss: 0.16269 | valid_rmse: 0.35182 |  0:06:03s
epoch 4  | loss: 0.10993 | valid_rmse: 0.30121 |  0:07:48s
epoch 5  | loss: 0.08882 | valid_rmse: 0.27747 |  0:09:30s
epoch 6  | loss: 0.0786  | valid_rmse: 0.26548 |  0:11:14s
epoch 7  | loss: 0.07187 | valid_rmse: 0.25915 |  0:12:59s
epoch 8  | loss: 0.06721 | valid_rmse: 0.2543  |  0:14:43s
epoch 9  | loss: 0.06388 | valid_rmse: 0.24759 |  0:16:28s
epoch 10 | loss: 0.06164 | valid_rmse: 0.24414 |  0:18:14s
epoch 11 | loss: 0.05993 | valid_rmse: 0.24125 |  0:20:02s
epoch 12 | loss: 0.05859 | valid_rmse: 0.23918 |  0:21:53s
epoch 13 | loss: 0.0575  | valid_rmse: 0.23723 |  0:23:43s
epoch 14 | loss: 0.0565  | valid_rmse: 0.23587 |  1:31:50s
epoch 15 | loss: 0.05577 | valid_rmse: 0.2348  |  1:33:21s
epoch 16 | loss: 0.05516 | valid_rmse: 0.23381 |  1:34:4

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


In [20]:
# 평가 지표
val_prediction = tabnet_model.predict(X_valid).reshape(-1)


def numerai_corr_arrays(era_arr, target_arr, pred_arr):
    df_eval = pd.DataFrame({
        "era": era_arr,
        "target": target_arr,
        "prediction": pred_arr
    })
    
    def era_corr(sub):
        return np.corrcoef(
            sub["prediction"].rank(pct=True),
            sub["target"]
        )[0, 1]
    
    return df_eval.groupby("era").apply(era_corr)


per_era_corr = numerai_corr_arrays(
    val_era,
    val_target,
    val_prediction
)

mean_corr = per_era_corr.mean()
std_corr = per_era_corr.std()
sharpe = mean_corr / std_corr

rmse = np.sqrt(mean_squared_error(val_target, val_prediction))
mae = mean_absolute_error(val_target, val_prediction)

print("\n========== TabNet Validation Result ==========")
print(f"RMSE: {rmse:.6f}")
print(f"MAE: {mae:.6f}")
print(f"per-era CORR mean: {mean_corr:.6f}")
print(f"per-era CORR std:  {std_corr:.6f}")
print(f"Sharpe:            {sharpe:.6f}")

display(per_era_corr.describe())



========== TabNet Validation Result ==========
RMSE: 0.231227
MAE: 0.169137
per-era CORR mean: 0.000518
per-era CORR std:  0.013690
Sharpe:            0.037820


count    23.000000
mean      0.000518
std       0.013690
min      -0.022497
25%      -0.007326
50%       0.001122
75%       0.012065
max       0.020173
dtype: float64